In [1]:
import os
import json
import time
from openai import OpenAI

API_KEY = "sk-proj-xxxxxxxxxxx" 
client = OpenAI(api_key=API_KEY)

In [2]:
# to avoid API failures

def generate_completion(model, system_prompt, user_content):
    
    """Helper to handle API calls with basic retry logic"""
    
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_content}
                ],

                # for stable reasoning
                temperature=0.0
            )
            return response.choices[0].message.content
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2) # Wait 2s before retry
            else:
                print(f"  [!] Failed after {max_retries} attempts: {e}")
                return None

In [3]:
# API connection test
response = client.responses.create(
    model="gpt-4.1-mini",
    input="Say hello in one word"
)

print(response.output_text)

Hello!


In [4]:
# ENRON (Full Training Set)
INPUT_FILE = "phase1_enron_clean_train.jsonl"
OUTPUT_FILE = "phase1b_enron_train_reasoning.jsonl"
MODEL = "gpt-4.1-mini"

SYSTEM_PROMPT = """You are an Executive Assistant Agent.
Task: Read the email and classify the Intent (e.g., Schedule Meeting, Request Info, Status Update).
Output Format:
Thought: <Analyze the sender's tone and request. Identify key dates/entities.>
Action: <The tool call, e.g., schedule_meeting(time="...") or None>
"""

In [5]:
print(f"Starting Enron Processing with {MODEL}...")

if os.path.exists(INPUT_FILE):
    with open(INPUT_FILE, 'r') as f_in, open(OUTPUT_FILE, 'w') as f_out:
        lines = f_in.readlines()
        print(f"  -> Found {len(lines)} emails to process.")
        
        for i, line in enumerate(lines):
            data = json.loads(line)
            
            output = generate_completion(MODEL, SYSTEM_PROMPT, data['input'])
            
            if output:
                data['output'] = output
                f_out.write(json.dumps(data) + "\n")
            
            if (i+1) % 10000 == 0: 
                print(f"  [Enron] Processed {i+1}/{len(lines)}")

    print(f"Enron Complete! Saved to {OUTPUT_FILE}")
else:
    print(f"Error: {INPUT_FILE} not found.")

Starting Enron Processing with gpt-4.1-mini...
  -> Found 209119 emails to process.
  [!] Failed after 3 attempts: Error code: 429 - {'error': {'message': 'Request too large for gpt-4.1-mini-long-context in organization org-szTK5N8e9P1j1kg8IOSCc2q7 on tokens per min (TPM): Limit 400000, Requested 424391. The input or output tokens must be reduced in order to run successfully. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}
  [Enron] Processed 10000/209119
  [Enron] Processed 20000/209119
  [Enron] Processed 30000/209119
  [Enron] Processed 40000/209119
  [Enron] Processed 50000/209119
  [Enron] Processed 60000/209119
  [Enron] Processed 70000/209119
  [Enron] Processed 80000/209119
  [Enron] Processed 90000/209119
  [Enron] Processed 100000/209119
  [Enron] Processed 110000/209119
  [Enron] Processed 120000/209119
  [Enron] Processed 130000/209119
  [Enron] Processed 140000/209119
  [Enron] Processe